# Extract `.tex` files from tarballs in Google Drive

This notebook (made for **Google Colab**) scans a Drive folder for `.tar.gz` / `.tgz` files and, for each one:

1. Extracts **only** the `.tex` files.
2. If there are several, concatenates them: the **main** file first (the one with `\documentclass`, preferring `main.tex`), then the rest in **alphabetical** order.
3. Saves one file per tarball to `<BASE_FOLDER>/latex-files/` named `tarballname-texname.tex`.
4. Records a status: **ok** (extracted), **none** (no `.tex` inside, skipped), **error** (archive could not be opened/extracted).
5. Verifies one output per successful tarball and writes `<BASE_FOLDER>/report/report.md`.

Run the cells top to bottom. Each `%%bash` cell runs in its own shell, so the settings are saved to `/content/tex_config.env` and read by every later cell.

## 1. Connect to Google Drive
Mounting Drive needs Colab's Python API, so this one cell is Python; everything else is bash.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Settings
Edit `BASE_FOLDER` (and `SOURCE_FOLDER` if the tarballs live elsewhere), then run.

In [ ]:
%%bash
# ============================================================
#  EDIT THESE VARIABLES, then run this cell
# ============================================================
cat > /content/tex_config.env <<'EOF'
# Base folder in Google Drive (everything is created under it)
BASE_FOLDER="/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426"

# Folder to explore for tarballs (.tar.gz / .tgz)
SOURCE_FOLDER="${BASE_FOLDER}"

# How deep to search: 1 = only that folder, 2+ = include subfolders
SEARCH_DEPTH=1

# Outputs
OUTPUT_FOLDER="${BASE_FOLDER}/latex-files"
REPORT_FOLDER="${BASE_FOLDER}/report"

# Local scratch space (not in Drive)
WORK_DIR="/content/tex_work"
EOF

source /content/tex_config.env
echo "Config saved to /content/tex_config.env"
echo "  BASE_FOLDER   = $BASE_FOLDER"
echo "  SOURCE_FOLDER = $SOURCE_FOLDER (depth $SEARCH_DEPTH)"
echo "  OUTPUT_FOLDER = $OUTPUT_FOLDER"
echo "  REPORT_FOLDER = $REPORT_FOLDER"

Config saved to /content/tex_config.env
  BASE_FOLDER   = /content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426
  SOURCE_FOLDER = /content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426 (depth 1)
  OUTPUT_FOLDER = /content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/latex-files
  REPORT_FOLDER = /content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/report


## 3. Explore the folder

In [ ]:
%%bash
source /content/tex_config.env
if [ ! -d "$SOURCE_FOLDER" ]; then
  echo "ERROR: folder not found: $SOURCE_FOLDER"
  echo "Check BASE_FOLDER / SOURCE_FOLDER in the config cell and that Drive is mounted."
  exit 1
fi
count=$(find "$SOURCE_FOLDER" -maxdepth "$SEARCH_DEPTH" -type f \( -iname '*.gz' -o -iname '*.tgz' \) | wc -l)
echo "Tarballs found in $SOURCE_FOLDER: $count"
echo "First 20:"
find "$SOURCE_FOLDER" -maxdepth "$SEARCH_DEPTH" -type f \( -iname '*.gz' -o -iname '*.tgz' \) | LC_ALL=C sort | head -n 20

Tarballs found in /content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426: 100
First 20:
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00001.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00002.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00004.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00005.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00006.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00007.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/2604.00009.tar.gz
/content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYEC

## 4. Extract `.tex` files
Writes the final files to `latex-files/` and keeps the per-tarball results in `$WORK_DIR/results.tsv`.

In [ ]:
%%bash
set -uo pipefail
source /content/tex_config.env

mkdir -p "$OUTPUT_FOLDER" "$REPORT_FOLDER"
rm -rf "$WORK_DIR" && mkdir -p "$WORK_DIR"
RESULTS="$WORK_DIR/results.tsv"
# internal columns (1-6 unchanged; 7-8 are new and go at the end):
# original, tex_count, tex_names, status, output_file, message, bib_bbl_count, bib_bbl_names
printf 'original\ttex_count\ttex_names\tstatus\toutput\tmessage\tbib_bbl_count\tbib_bbl_names\n' > "$RESULTS"

clean() { printf '%s' "$1" | tr '\t\n\r' '   '; }
# empty fields are stored as "-" so tab-separated columns never collapse
record() {
  printf '%s\t%s\t%s\t%s\t%s\t%s\t%s\t%s\n' \
    "$(clean "$1")" "$2" "$(clean "${3:--}")" "$4" "$(clean "${5:--}")" "$(clean "${6:--}")" \
    "${7:-0}" "$(clean "${8:--}")" >> "$RESULTS"
}
# list of paths -> "a.tex; sec/b.tex" (leading ./ removed)
fmt_names() { [ -s "$1" ] && sed 's#^\./##' "$1" | paste -sd ';' | sed 's/;/; /g'; }

# Pick the main .tex: prefer files containing \documentclass; among them one
# named main.tex, else the first alphabetically. If none has \documentclass,
# use main.tex if present, else the first alphabetically.
choose_main() {
  local dir="$1" list="$2" cands m
  cands=$(while IFS= read -r p; do
            grep -qE '^[^%]*\\documentclass' "$dir/$p" 2>/dev/null && printf '%s\n' "$p"
          done < "$list")
  [ -z "$cands" ] && cands=$(cat "$list")
  m=$(printf '%s\n' "$cands" | awk -F/ 'tolower($NF)=="main.tex"' | head -n1)
  [ -z "$m" ] && m=$(printf '%s\n' "$cands" | head -n1)
  printf '%s' "$m"
}

total=0
while IFS= read -r -d '' tarball; do
  total=$((total+1))
  name=$(basename "$tarball")
  base="$name"
  case "$base" in
    *.[tT][aA][rR].[gG][zZ]) base="${base%.*}"; base="${base%.*}" ;;
    *.[tT][gG][zZ])          base="${base%.*}" ;;
  esac
  tmp=$(mktemp -d "$WORK_DIR/x.XXXXXX")

  # 1) list contents (fails -> error)
  if ! tar -tzf "$tarball" > "$tmp.all" 2> "$tmp.err"; then
    record "$name" 0 "" "error" "" "cannot open: $(head -c 200 "$tmp.err")"
    echo "[error] $name"
    rm -rf "$tmp" "$tmp".*; continue
  fi

  # 2) .tex list (decides the status) + .bib/.bbl list, each alphabetical
  # extension match is case-insensitive (.tex/.TEX, .bib/.BIB, .bbl/.BBL)
  grep -iE '\.tex$' "$tmp.all" | LC_ALL=C sort -u > "$tmp.list" || true
  grep -iE '\.bib$' "$tmp.all" | LC_ALL=C sort -u >  "$tmp.extra" || true
  grep -iE '\.bbl$' "$tmp.all" | LC_ALL=C sort -u >> "$tmp.extra" || true
  n=$(wc -l < "$tmp.list" | tr -d ' ')
  nx=$(wc -l < "$tmp.extra" | tr -d ' ')
  if [ "$n" -eq 0 ]; then
    # no .tex -> skip, even if .bib/.bbl exist (they are only listed in the report)
    record "$name" 0 "" "none" "" "no .tex files in archive" "$nx" "$(fmt_names "$tmp.extra")"
    echo "[none ] $name"
    rm -rf "$tmp" "$tmp".*; continue
  fi

  # 3) extract only the .tex, .bib and .bbl files
  cat "$tmp.list" "$tmp.extra" > "$tmp.extract"
  if ! tar -xzf "$tarball" -C "$tmp" -T "$tmp.extract" 2> "$tmp.err"; then
    record "$name" "$n" "$(fmt_names "$tmp.list")" "error" "" "extract failed: $(head -c 200 "$tmp.err")" "$nx" "$(fmt_names "$tmp.extra")"
    echo "[error] $name"
    rm -rf "$tmp" "$tmp".*; continue
  fi
  missing=0
  while IFS= read -r p; do [ -f "$tmp/$p" ] || missing=1; done < "$tmp.extract"
  if [ "$missing" -eq 1 ]; then
    record "$name" "$n" "$(fmt_names "$tmp.list")" "error" "" "some .tex/.bib/.bbl entries missing after extraction" "$nx" "$(fmt_names "$tmp.extra")"
    echo "[error] $name"
    rm -rf "$tmp" "$tmp".*; continue
  fi

  # 4) order: main .tex, other .tex alphabetically, then .bib, then .bbl (always last)
  main=$(choose_main "$tmp" "$tmp.list")
  { printf '%s\n' "$main"; grep -vxF -- "$main" "$tmp.list" || true; } > "$tmp.order"
  cat "$tmp.order" "$tmp.extra" > "$tmp.concat"
  out="$OUTPUT_FOLDER/${base}-$(basename "$main")"

  # 5) write one final .tex (copy if it is the only file, otherwise concatenate)
  ok=1
  if [ $((n+nx)) -eq 1 ]; then
    cp "$tmp/$main" "$out.part" || ok=0
  else
    { while IFS= read -r p; do
        printf '%% ===== FILE: %s =====\n' "$p"
        cat "$tmp/$p"
        printf '\n'
      done < "$tmp.concat"; } > "$out.part" || ok=0
  fi
  if [ "$ok" -eq 1 ] && mv -f "$out.part" "$out"; then
    record "$name" "$n" "$(fmt_names "$tmp.order")" "ok" "$(basename "$out")" "" "$nx" "$(fmt_names "$tmp.extra")"
    echo "[ok   ] $name -> $(basename "$out") ($n tex, $nx bib/bbl)"
  else
    rm -f "$out.part"
    record "$name" "$n" "$(fmt_names "$tmp.order")" "error" "" "could not write output" "$nx" "$(fmt_names "$tmp.extra")"
    echo "[error] $name"
  fi
  rm -rf "$tmp" "$tmp".*
done < <(find "$SOURCE_FOLDER" -maxdepth "$SEARCH_DEPTH" -type f \( -iname '*.tar.gz' -o -iname '*.tgz' \) -print0 | LC_ALL=C sort -z)

echo
echo "Processed: $total"
awk -F'\t' 'NR>1{c[$4]++; x+=$7} END{printf "ok: %d | none: %d | error: %d | bib/bbl files found: %d\n", c["ok"]+0, c["none"]+0, c["error"]+0, x}' "$RESULTS"

[ok   ] 2604.00001.tar.gz -> 2604.00001-neurips.tex (7 tex, 1 bib/bbl)
[ok   ] 2604.00002.tar.gz -> 2604.00002-neurips_2024.tex (1 tex, 1 bib/bbl)
[ok   ] 2604.00004.tar.gz -> 2604.00004-main.tex (1 tex, 1 bib/bbl)
[ok   ] 2604.00005.tar.gz -> 2604.00005-main.tex (11 tex, 1 bib/bbl)
[ok   ] 2604.00006.tar.gz -> 2604.00006-template.tex (1 tex, 1 bib/bbl)
[error] 2604.00007.tar.gz
[ok   ] 2604.00009.tar.gz -> 2604.00009-eyla_paper.tex (1 tex, 0 bib/bbl)
[ok   ] 2604.00010.tar.gz -> 2604.00010-iclr2026_conference.tex (2 tex, 1 bib/bbl)
[ok   ] 2604.00011.tar.gz -> 2604.00011-manuscript.tex (1 tex, 1 bib/bbl)
[ok   ] 2604.00012.tar.gz -> 2604.00012-SafeReAct_copy.tex (2 tex, 2 bib/bbl)
[ok   ] 2604.00013.tar.gz -> 2604.00013-arxiv.tex (1 tex, 1 bib/bbl)
[ok   ] 2604.00014.tar.gz -> 2604.00014-main.tex (1 tex, 2 bib/bbl)
[ok   ] 2604.00015.tar.gz -> 2604.00015-template.tex (1 tex, 1 bib/bbl)
[ok   ] 2604.00016.tar.gz -> 2604.00016-cogsci.tex (1 tex, 1 bib/bbl)
[ok   ] 2604.00018.tar.gz -> 2

## Generate `report.md`

In [ ]:
!rm -rf "$WORK_DIR"

In [ ]:
%%bash
source /content/tex_config.env
RESULTS="$WORK_DIR/results.tsv"
VER="$WORK_DIR/verification.txt"

# results.tsv only exists after the extraction cell has run in this runtime
if [ ! -f "$RESULTS" ]; then
  echo "No results found in $WORK_DIR."
  echo "Run the extraction cell first (results are lost if the Colab runtime restarted)."
  exit 0
fi

# results.tsv must come from the NEW extraction cell (8 columns incl. bib/bbl)
cols=$(awk -F'\t' 'NR==1{print NF; exit}' "$RESULTS")
if [ "$cols" -lt 8 ]; then
  echo "results.tsv has $cols columns (old extraction cell): no .bib/.bbl data in it."
  echo "Replace the extraction cell with the new version, run it, then run this cell again."
  exit 0
fi

# Rebuild the verification here if it is missing or older than the results
# (the extraction cell deletes WORK_DIR, so an old verification never survives a re-run)
if [ ! -f "$VER" ] || [ "$RESULTS" -nt "$VER" ]; then
  echo "Verification missing or out of date -> recomputing it now."
  total=$(awk 'NR>1' "$RESULTS" | wc -l | tr -d ' ')
  ok=$(awk -F'\t' 'NR>1 && $4=="ok"' "$RESULTS" | wc -l | tr -d ' ')
  none=$(awk -F'\t' 'NR>1 && $4=="none"' "$RESULTS" | wc -l | tr -d ' ')
  error=$(awk -F'\t' 'NR>1 && $4=="error"' "$RESULTS" | wc -l | tr -d ' ')
  present=0; problems=""
  while IFS= read -r f; do
    if [ -s "$OUTPUT_FOLDER/$f" ]; then present=$((present+1)); else problems+="  - missing/empty: $f"$'\n'; fi
  done < <(awk -F'\t' 'NR>1 && $4=="ok"{print $5}' "$RESULTS")
  unique=$(awk -F'\t' 'NR>1 && $4=="ok"{print $5}' "$RESULTS" | sort -u | wc -l | tr -d ' ')
  [ "$unique" -ne "$ok" ] && problems+="  - $((ok-unique)) output name collision(s)"$'\n'
  in_folder=$(find "$OUTPUT_FOLDER" -maxdepth 1 -type f -name '*.tex' | wc -l | tr -d ' ')
  if [ "$present" -eq "$ok" ] && [ "$unique" -eq "$ok" ] && [ $((ok+none+error)) -eq "$total" ]; then result="PASS"; else result="FAIL"; fi
  printf 'result=%s\ntotal=%s\nok=%s\nnone=%s\nerror=%s\npresent=%s\nunique=%s\nin_folder=%s\n' \
    "$result" "$total" "$ok" "$none" "$error" "$present" "$unique" "$in_folder" > "$VER"
  printf '%s' "$problems" > "$WORK_DIR/problems.txt"
fi
source "$VER"
REPORT="$REPORT_FOLDER/report.md"
mkdir -p "$REPORT_FOLDER"

esc() { printf '%s' "$1" | sed 's/|/\\|/g'; }

# bib/bbl check: every .bib/.bbl recorded for an ok tarball must appear as a
# "% ===== FILE: ... =====" block inside its final .tex
bib_ok=0; bib_bad=0; bib_problems=""
while IFS=$'\t' read -r orig out xcnt; do
  [ "$xcnt" -gt 0 ] || continue
  found=$(grep -ciE '^% ===== FILE: .*\.(bib|bbl) =====$' "$OUTPUT_FOLDER/$out" 2>/dev/null)
  if [ "${found:-0}" -eq "$xcnt" ]; then bib_ok=$((bib_ok+1))
  else bib_bad=$((bib_bad+1)); bib_problems+="  - $orig: expected $xcnt bib/bbl block(s) in $out, found ${found:-0}"$'\n'; fi
done < <(awk -F'\t' 'NR>1 && $4=="ok"{print $1"\t"$5"\t"$7}' "$RESULTS")
[ "$bib_bad" -eq 0 ] && bib_result="PASS" || bib_result="FAIL"

with_bib=$(awk -F'\t' 'NR>1 && $4=="ok" && $7>0' "$RESULTS" | wc -l | tr -d ' ')
bib_total=$(awk -F'\t' 'NR>1 && $4=="ok"{x+=$7} END{print x+0}' "$RESULTS")

{
  echo "# LaTeX extraction report"
  echo
  echo "- **Date:** $(date '+%Y-%m-%d %H:%M:%S %Z')"
  echo "- **Source folder:** \`$SOURCE_FOLDER\` (depth $SEARCH_DEPTH)"
  echo "- **Output folder:** \`$OUTPUT_FOLDER\`"
  echo "- **Concatenation order:** main .tex → other .tex (alphabetical) → .bib → .bbl"
  echo
  echo "## Summary"
  echo
  echo "| Status | Count |"
  echo "|---|---:|"
  echo "| ok | $ok |"
  echo "| none | $none |"
  echo "| error | $error |"
  echo "| **Total tarballs** | **$total** |"
  echo
  echo "- ok tarballs that included .bib/.bbl: $with_bib ($bib_total files appended)"
  echo
  echo "## Verification"
  echo
  echo "- Successful extractions (ok): $ok"
  echo "- Final .tex files present for them: $present"
  echo "- Unique output names: $unique"
  echo "- Result: **$result** (one final .tex per successful tarball)"
  echo "- .bib/.bbl appended correctly: $bib_ok of $with_bib tarballs → **$bib_result**"
  if [ -s "$WORK_DIR/problems.txt" ]; then echo; echo "Problems:"; cat "$WORK_DIR/problems.txt"; fi
  if [ -n "$bib_problems" ]; then echo; echo "bib/bbl problems:"; printf '%s' "$bib_problems"; fi
  echo
  echo "## Details"
  echo
  echo "| Original | tex files | tex names | bib/bbl files | bib/bbl names | status |"
  echo "|---|---:|---|---:|---|---|"
  awk -F'\t' 'NR>1' "$RESULTS" | while IFS=$'\t' read -r orig cnt names status out msg xcnt xnames; do
    echo "| $(esc "$orig") | $cnt | $(esc "$names") | ${xcnt:-0} | $(esc "${xnames:--}") | $status |"
  done
  if [ "$error" -gt 0 ]; then
    echo
    echo "## Error details"
    echo
    echo "| Original | Message |"
    echo "|---|---|"
    awk -F'\t' 'NR>1 && $4=="error"' "$RESULTS" | while IFS=$'\t' read -r orig cnt names status out msg xcnt xnames; do
      echo "| $(esc "$orig") | $(esc "$msg") |"
    done
  fi
} > "$REPORT"

echo "Report written to: $REPORT"
echo "Tarballs: $total | ok: $ok | none: $none | error: $error | verification: $result"
echo "bib/bbl: $with_bib ok tarballs with $bib_total files appended | check: $bib_result"
[ -n "$bib_problems" ] && printf '%s' "$bib_problems"
echo
echo "Tarballs with .bib/.bbl:"
awk -F'\t' 'NR>1 && $7>0{printf "  %-30s %s  [%s]\n", $1, $8, $4}' "$RESULTS" | head -n 50
[ "$(awk -F'\t' 'NR>1 && $7>0' "$RESULTS" | wc -l)" -gt 50 ] && echo "  ... (full list in report.md)"
true

Verification missing or out of date -> recomputing it now.
Report written to: /content/drive/MyDrive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-0426/report/report.md
Tarballs: 100 | ok: 99 | none: 0 | error: 1 | verification: PASS
bib/bbl: 75 ok tarballs with 88 files appended | check: PASS

Tarballs with .bib/.bbl:
  2604.00001.tar.gz              ref.bib  [ok]
  2604.00002.tar.gz              references.bib  [ok]
  2604.00004.tar.gz              references.bib  [ok]
  2604.00005.tar.gz              components/refer.bib  [ok]
  2604.00006.tar.gz              references.bib  [ok]
  2604.00010.tar.gz              references.bib  [ok]
  2604.00011.tar.gz              bibliography.bib  [ok]
  2604.00012.tar.gz              normal_generated_py3.bib; neurips_2025.bbl  [ok]
  2604.00013.tar.gz              sample-base.bib  [ok]
  2604.00014.tar.gz              amia.bib; main.bbl  [ok]
  2604.00015.tar.gz              references.bib  [ok]
  2604.00016.tar.gz        